# Hands-on Exercise 3 — Log a Model with a Signature
### AI Operations (AIOps) — MLflow Deep Dive | ~10–15 minutes

**Referenced in:** *MLflow Deep Dive Slide Deck*, Section 3 (MLflow Models)

**Objective:** attach a proper input/output **signature** and **input example** to a logged model,
then load it back with the framework-agnostic `pyfunc` interface and run inference on new data.

**Steps (from the slide deck):**
1. Using your Exercise 1 model, generate predictions on a small sample of training data.
2. Call `infer_signature()` to build a signature from that sample.
3. Re-log the model with `mlflow.sklearn.log_model(..., signature=..., input_example=...)`.
4. Open the run in the MLflow UI and confirm the signature appears under the Artifacts tab (`MLmodel` file).
5. Load the model back with `mlflow.pyfunc.load_model()` in a fresh Python session and call `.predict()` on new data.

**Deliverable:** a run containing a model artifact with a valid signature, plus a short script proving
`pyfunc.load_model()` + `.predict()` works end-to-end.

> **Prerequisite:** the MLflow Tracking Server must still be running at `http://localhost:5000`.

## Step 0 — Setup

In [ ]:
# !pip install mlflow scikit-learn pandas --quiet
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("iris-classifier")

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 1 — Train a model and generate sample predictions

In [ ]:
model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
model.fit(X_train, y_train)

sample_inputs = X_train.iloc[:5]
sample_predictions = model.predict(sample_inputs)
print(sample_inputs)
print("Sample predictions:", sample_predictions)

## Step 2 — Build a signature with `infer_signature()`
The signature records the expected input schema (column names + dtypes) and the output schema.

In [ ]:
signature = infer_signature(sample_inputs, sample_predictions)
print(signature)

## Step 3 — Log the model WITH the signature and an input example

In [ ]:
with mlflow.start_run(run_name="rf-with-signature") as run:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("accuracy", acc)

    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=sample_inputs,
    )

    signed_run_id = run.info.run_id

print(f"Logged run with signature: {signed_run_id}  (accuracy={acc:.4f})")

## Step 4 — Inspect the signature in the MLflow UI
1. Open **http://localhost:5000** → **iris-classifier** experiment → the `rf-with-signature` run.
2. Open the **Artifacts** tab → `model` → view the `MLmodel` file.
3. Confirm you can see a `signature:` block listing `inputs` and `outputs`.

You can also inspect it directly from Python without leaving the notebook:

In [ ]:
from mlflow.models import get_model_info

model_uri = f"runs:/{signed_run_id}/model"
info = get_model_info(model_uri)
print("Signature recorded in MLmodel file:")
print(info.signature)

## Step 5 — Load the model with `pyfunc` and predict on new data
This simulates a *fresh Python session* consuming the model purely through its MLflow URI — no need to import scikit-learn model internals.

In [ ]:
import mlflow.pyfunc

loaded_model = mlflow.pyfunc.load_model(model_uri)

new_data = X_test.iloc[:8]
predictions = loaded_model.predict(new_data)
print("Predictions from the reloaded pyfunc model:")
print(predictions)

### Try it: what happens if the input doesn't match the signature?
Uncomment and run the cell below to see MLflow's schema validation in action (this previews Exercise 5's serving validation behaviour).

In [ ]:
# bad_input = new_data.rename(columns={"sepal length (cm)": "sepal_length"})  # wrong column name
# loaded_model.predict(bad_input)  # -> raises a schema validation error

---
### ✅ Deliverable checklist
- [ ] A run (`rf-with-signature`) whose logged model has a non-empty `signature` in its `MLmodel` file
- [ ] An `input_example` visible alongside the model artifact
- [ ] A working call to `mlflow.pyfunc.load_model(...)` followed by `.predict()` on new data, executed in this notebook
- [ ] (Optional) A screenshot of the signature block from the MLflow UI